<a href="https://colab.research.google.com/github/IsaacMh/prediccion-de-abandono-y-exito-academico/blob/main/Prediccion_del_abandono_y_exito_escolar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Librerías necesarias

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

###Lectura de datos

In [4]:
url = "https://drive.google.com/uc?id=1JmzingmKDxcQizwszGAD9SxcTFENXKR2"
data = pd.read_csv(url)

In [5]:
print(data.shape)
data.head()

(4424, 35)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


##Tipo de datos


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 35 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Nacionality                                     4424 non-null   int64  
 7   Mother's qualification                          4424 non-null   int64  
 8   Father's qualification                          4424 non-null   int64  
 9   Mother's occupation                      

### Limpieza de datos

####Ver si hay valores nulos o filas duplicadas

In [7]:
#Comprobar si hay valores vacios/ nulos
print(data.isnull().sum())

#Filas duplicadas
print("Duplicados:", data.duplicated().sum())

Marital status                                    0
Application mode                                  0
Application order                                 0
Course                                            0
Daytime/evening attendance                        0
Previous qualification                            0
Nacionality                                       0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Displaced                                         0
Educational special needs                         0
Debtor                                            0
Tuition fees up to date                           0
Gender                                            0
Scholarship holder                                0
Age at enrollment                                 0
International                                     0
Curricular u

####Eliminar filas no necesarias

In [8]:
data = data[data["Target"] != "Enrolled"]
data['Target'] = data['Target'].map({'Dropout': 1, 'Graduate': 0})
print(data["Target"].value_counts())


Target
0    2209
1    1421
Name: count, dtype: int64


####Eliminar columnas no necesarias

In [13]:
#Se elimina nacionalidad e international porque no hay variabilidad, la mayoria solo pertenece a 1 (portugues)
data = data.drop(columns=["Nacionality", "International"])
#Ver si el genero afecta al exito o abandono escolar, se concluye que si
data.groupby("Gender")["Target"].value_counts(normalize=True)

Gender  Target
0       0         0.697606
        1         0.302394
1       1         0.561249
        0         0.438751
Name: proportion, dtype: float64

In [17]:
print(data[["Unemployment rate", "Inflation rate", "GDP"]].nunique())
data.groupby("Target")[["Unemployment rate", "Inflation rate", "GDP"]].mean()
# se deduce que no influyen de forma significante en el target lo cual ocasionaria ruido, asi que se elimina

Unemployment rate    10
Inflation rate        9
GDP                  10
dtype: int64


,Unemployment rate,Inflation rate,GDP
Target,,,
0,11.639339,1.197918,0.081833
1,11.616397,1.283955,-0.150859


In [21]:
data = data.drop(columns=[
    "Unemployment rate",
    "Inflation rate",
    "GDP"
])
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3630 entries, 0 to 4423
Data columns (total 31 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  3630 non-null   int64  
 1   Application mode                                3630 non-null   int64  
 2   Application order                               3630 non-null   int64  
 3   Course                                          3630 non-null   int64  
 4   Daytime/evening attendance                      3630 non-null   int64  
 5   Previous qualification                          3630 non-null   int64  
 6   Mother's qualification                          3630 non-null   int64  
 7   Father's qualification                          3630 non-null   int64  
 8   Mother's occupation                             3630 non-null   int64  
 9   Father's occupation                           